# LandmarkDiff Quickstart

Predict post-operative facial appearance from a single photo.

This notebook runs through the core pipeline:
1. Extract 478 facial landmarks with MediaPipe
2. Apply procedure-specific deformations
3. Generate the predicted result

**Runtime:** Use a GPU runtime (T4 is fine) for the full pipeline, or CPU for TPS-only mode.

In [ ]:
# install landmarkdiff
!pip install -q git+https://github.com/dreamlessx/LandmarkDiff-public.git

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from PIL import Image

## 1. Load a face image

Upload your own or use a sample. Needs to be a clear frontal photo with good lighting.

In [ ]:
# upload your own image, or download a sample
import urllib.request
from pathlib import Path

# sample from FFHQ (thispersondoesnotexist-style)
sample_url = (
    "https://raw.githubusercontent.com/NVlabs/ffhq-dataset/master/thumbnails128x128/00000.png"
)
sample_path = Path("sample_face.png")

if not sample_path.exists():
    urllib.request.urlretrieve(sample_url, sample_path)

img = Image.open(sample_path).convert("RGB").resize((512, 512))
display(img)

## 2. Extract landmarks

MediaPipe detects 478 facial landmarks in 3D. This runs on CPU.

In [ ]:
from landmarkdiff.landmarks import extract_landmarks, render_landmark_image

img_array = np.array(img)
landmarks = extract_landmarks(img_array)

if landmarks is None:
    print("no face detected - try a different photo")
else:
    print(f"detected {len(landmarks.landmarks)} landmarks")
    print(f"confidence: {landmarks.confidence:.2f}")

    # draw the mesh
    mesh_img = render_landmark_image(landmarks, 512, 512)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_array)
    axes[0].set_title("input")
    axes[0].axis("off")
    axes[1].imshow(mesh_img)
    axes[1].set_title("478-point face mesh")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

## 3. Deform landmarks

Apply a procedure-specific deformation. Each procedure moves specific landmarks by calibrated displacement vectors.

In [ ]:
from landmarkdiff.manipulation import apply_procedure_preset

procedure = "rhinoplasty"  # try: blepharoplasty, rhytidectomy, orthognathic, brow_lift, mentoplasty
intensity = 60  # 0 to 100

deformed = apply_procedure_preset(landmarks, procedure, intensity=intensity)

# show original vs deformed mesh
mesh_original = render_landmark_image(landmarks, 512, 512)
mesh_deformed = render_landmark_image(deformed, 512, 512)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(mesh_original)
axes[0].set_title("original mesh")
axes[0].axis("off")
axes[1].imshow(mesh_deformed)
axes[1].set_title(f"deformed ({procedure} @ {intensity}%)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Generate conditioning image

The deformed mesh gets rendered as a tessellation wireframe that ControlNet uses as its conditioning signal.

In [ ]:
from landmarkdiff.conditioning import generate_conditioning

landmark_img, canny_img, wireframe = generate_conditioning(deformed, img_array, 512, 512)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(landmark_img)
axes[0].set_title("landmarks")
axes[0].axis("off")
axes[1].imshow(wireframe, cmap="gray")
axes[1].set_title("tessellation wireframe")
axes[1].axis("off")
axes[2].imshow(canny_img, cmap="gray")
axes[2].set_title("canny edges")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 5. Generate surgical mask

A feathered mask restricts the generation to the surgical region.

In [ ]:
from landmarkdiff.masking import generate_surgical_mask

mask = generate_surgical_mask(landmarks, procedure, 512, 512)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(mask, cmap="gray")
axes[0].set_title(f"surgical mask ({procedure})")
axes[0].axis("off")

# overlay mask on input
overlay = img_array.copy().astype(float)
overlay[..., 0] = overlay[..., 0] * (1 - 0.3 * mask) + 255 * 0.3 * mask
axes[1].imshow(overlay.astype(np.uint8))
axes[1].set_title("mask overlay")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6. TPS warp (CPU, no GPU needed)

The simplest prediction mode - just geometrically warps pixels. Fast but not photorealistic.

In [ ]:
from landmarkdiff.synthetic.tps_warp import warp_image_tps

# get pixel coordinates
src = landmarks.pixel_coords[:, :2].copy()
dst = deformed.pixel_coords[:, :2].copy()

# scale to 512x512
src[:, 0] *= 512 / landmarks.image_width
src[:, 1] *= 512 / landmarks.image_height
dst[:, 0] *= 512 / deformed.image_width
dst[:, 1] *= 512 / deformed.image_height

warped = warp_image_tps(img_array, src, dst)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_array)
axes[0].set_title("original")
axes[0].axis("off")
axes[1].imshow(warped)
axes[1].set_title(f"TPS warp ({procedure} @ {intensity}%)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 7. Full pipeline (GPU)

The full ControlNet pipeline produces photorealistic results. Needs a GPU with ~6GB VRAM.

**Note:** This downloads ~6GB of model weights on first run.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("no GPU available - switch to a GPU runtime for this cell")
    print("the TPS warp above works on CPU though")
else:
    from landmarkdiff.inference import LandmarkDiffPipeline

    pipeline = LandmarkDiffPipeline(mode="controlnet", device="cuda")
    pipeline.load()

    result = pipeline.generate(
        img_array,
        procedure=procedure,
        intensity=intensity,
        num_inference_steps=30,
        seed=42,
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(result["input"])
    axes[0].set_title("input")
    axes[0].axis("off")
    axes[1].imshow(result["conditioning"])
    axes[1].set_title("conditioning")
    axes[1].axis("off")
    axes[2].imshow(result["output"])
    axes[2].set_title("prediction")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

    if result.get("identity_check"):
        print(f"identity similarity: {result['identity_check']:.3f}")

## 8. Compare all procedures

Run all six procedure presets on the same face.

In [ ]:
procedures = ["rhinoplasty", "blepharoplasty", "rhytidectomy", "orthognathic", "brow_lift", "mentoplasty"]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(img_array)
axes[0].set_title("original")
axes[0].axis("off")

for i, proc in enumerate(procedures):
    deformed_p = apply_procedure_preset(landmarks, proc, intensity=60)
    src_p = landmarks.pixel_coords[:, :2].copy()
    dst_p = deformed_p.pixel_coords[:, :2].copy()
    src_p[:, 0] *= 512 / landmarks.image_width
    src_p[:, 1] *= 512 / landmarks.image_height
    dst_p[:, 0] *= 512 / deformed_p.image_width
    dst_p[:, 1] *= 512 / deformed_p.image_height

    warped_p = warp_image_tps(img_array, src_p, dst_p)
    axes[i + 1].imshow(warped_p)
    axes[i + 1].set_title(proc)
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

## Next steps

- Try different procedures and intensities
- Upload your own photos
- Check out the [Gradio demo](https://github.com/dreamlessx/LandmarkDiff-public#gradio-web-demo) for a full interactive UI
- See the [tutorials](https://github.com/dreamlessx/LandmarkDiff-public/tree/main/docs/tutorials) for custom procedures, training, and deployment